In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import  accuracy_score, f1_score, precision_score,confusion_matrix, recall_score, roc_auc_score
from sklearn.inspection import permutation_importance
from sklearn import metrics
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

In [ ]:

from google.colab import files

uploaded = files.upload()

In [ ]:

df = pd.read_csv('creditcard_2023.csv')
print(df.head(7))

In [ ]:
print(df.tail())
print(df['Class'].value_counts())

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:

df.info()

In [ ]:
df.value_counts()

In [ ]:
df.describe().T

In [ ]:
df['Class'].unique()


In [ ]:
df[df.duplicated()].count()


In [ ]:
df.isna().sum()

In [ ]:
sns.countplot(data=df, x='Class')

plt.savefig("equlib.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("equlib.jpg")

In [ ]:
df.shape
plt.figure(figsize=(10, 6))
sns.scatterplot(x='V1', y='V2', hue='Class', data=df.sample(frac=0.1, random_state=42), alpha=0.6)
plt.title('V1 vs V2,coloré par classe (données échantillonnées)')

plt.savefig("V1_V2.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("V1_V2.jpg")

In [ ]:
sns.boxplot(df, x="V1")

plt.savefig("box.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("box.jpg")

In [ ]:
sns.boxplot(df, x="V2")

plt.savefig("box2.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("box2.jpg")

In [ ]:
sns.boxplot(df, x="V3")
plt.show();

In [ ]:
sns.boxplot(df, x="V24")
plt.show();

In [ ]:
sns.boxplot(df, x="V28")
plt.savefig("box28.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("box28.jpg")

In [ ]:

correlation_matrix = df.corr()
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", cbar=True)
plt.title("Feature Correlation Matrix")
plt.savefig("corr.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("corr.jpg")


In [ ]:

df = df.drop(['id'], axis=1)

In [ ]:
X = df.drop(columns=['Class'])
y = df["Class"]

In [ ]:

seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state = seed)



In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=1,
    max_samples=0.7,
    bootstrap=True
)

In [ ]:
param_grid = {
    'n_estimators': [20, 50],
    'max_depth': [5, 10]
}

In [ ]:
grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=2,
    scoring='accuracy',
    n_jobs=1,
    verbose=2
)

In [ ]:
grid_rf.fit(X_train, y_train)
print("Meilleur paramètre :", grid_rf.best_params_)
print("Meilleur score CV :", grid_rf.best_score_)

best_model = grid_rf.best_estimator_
y_pred = best_model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print("Rapport de classification :")

In [ ]:
mse_test = MSE(y_test, y_pred)
print(mse_test)

In [ ]:
rmse_test = MSE(y_test, y_pred)**(1/2)
print(' RMSE  {:.2f}'.format(rmse_test))

In [ ]:
y_pred = best_model.predict(X_test)
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision :", precision_score(y_test, y_pred))
print("Recall :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

In [ ]:
best_rf = grid_rf.best_estimator_
importances_rf = pd.Series(
    best_rf.feature_importances_,
    index=X.columns)
sorted_importances_rf = importances_rf.sort_values()
sorted_importances_rf.plot(kind='barh', color='lightgreen')
plt.savefig("distv.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("distv.jpg")


In [ ]:

y_probs = best_model.predict(X_test)
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive ')
plt.ylabel('True Positive ')
plt.title('Receiver(ROC) Curve')
plt.legend(loc='lower right')
plt.savefig("roc.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("roc.jpg")

In [ ]:
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Fraud', 'Fraud'], yticklabels=['No Fraud', 'Fraud'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig("conf.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("conf.jpg")

In [ ]:
train_accuracy = accuracy_score(y_test, y_pred)
test_accuracy = accuracy_score(y_test, y_pred)
plt.figure(figsize=(6, 4))
accuracy_values = [train_accuracy, test_accuracy]
accuracy_labels = ['Train Accuracy', 'Test Accuracy']
plt.bar(accuracy_labels, accuracy_values, color=['blue', 'red'])
plt.ylabel('Accuracy')
plt.title('Train Accuracy vs Test Accuracy')
plt.ylim(0, 1)
plt.savefig("acufor.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("acufor.jpg")

In [ ]:


random_forest = grid_rf.best_estimator_
TOP_N = 20
mdi_importances = random_forest.feature_importances_
mdi_sorted_idx = np.argsort(mdi_importances)[-TOP_N:]

result = permutation_importance(
    random_forest,
    X_train,
    y_train,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_importances = result.importances_mean
perm_sorted_idx = np.argsort(perm_importances)[-TOP_N:]



In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10))


ax1.barh(
    range(len(mdi_sorted_idx)),
    mdi_importances[mdi_sorted_idx]
)

ax1.set_yticks(range(len(mdi_sorted_idx)))
ax1.set_yticklabels(X_train.columns[mdi_sorted_idx])
ax1.set_xlabel("Mean Decrease in Impurity (MDI)")
ax1.set_title("Top Feature Importance - MDI")

ax2.boxplot(
    result.importances[perm_sorted_idx].T,
    vert=False,
    labels=X_train.columns[perm_sorted_idx]
)

ax2.set_xlabel("Mean Decrease in Accuracy (MDA)")
ax2.set_title("Top Permutation Importance")

plt.tight_layout()
plt.savefig("bb.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("bb.jpg")

In [ ]:
param_grid_xgb = {
    'n_estimators': [20, 50],
    'max_depth': [5, 10]
    }

xgb = XGBClassifier(
    random_state=123,
    objective='binary:logistic'
    )
CV_xgb = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=2,
    scoring='accuracy',
    n_jobs=1,
    verbose=2
)

In [ ]:

le = LabelEncoder()
y_train = le.fit_transform(y_train)

y_test = le.fit_transform(y_test)

In [ ]:

model_xgb = CV_xgb.fit(X_train, y_train)

In [ ]:
CV_xgb.get_params

In [ ]:

model_xgb = XGBClassifier(
    n_estimators=500,
    random_state=123,
    objective='binary:logistic',
    max_depth=6,
    colsample_bylevel=1,
    colsample_bytree=1,
    min_child_weight=3,
    learning_rate=0.01,
    reg_lambda=1,
    min_split_loss=0.2,
    reg_alpha=0)

model_xgb.fit(X_train, y_train)
y_pred_xgb = model_xgb.predict(X_test)

print("Accuracy xgboost {:0.2f}%.".format(accuracy_score(y_test, y_pred_xgb) * 100))

In [ ]:
roc_xgb = roc_auc_score(y_test, y_pred_xgb)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

model_results_xgb = pd.DataFrame([['Xgboost', acc_xgb, prec_xgb, rec_xgb, f1_xgb, roc_xgb]],
                              columns = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC'])
model_results_xgb

In [ ]:
matrix_xgb = confusion_matrix(y_test, y_pred_xgb)
print(matrix_xgb)

plt.figure(figsize=(8, 6))
sns.heatmap(matrix_xgb, annot=True, fmt='d', cmap='Blues', xticklabels=['No Fraud', 'Fraud'], yticklabels=['No Fraud', 'Fraud'])
plt.title('matrice de confusion - XGBoost')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.savefig("confxg.jpg", dpi=300, bbox_inches="tight")
plt.show()

from google.colab import files
files.download("confxg.jpg")

In [ ]:
train_accuracy = accuracy_score(y_test, y_pred)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
plt.figure(figsize=(6, 4))
accuracy_values = [train_accuracy, acc_xgb]
accuracy_labels = ['Train Accuracy', 'Test Accuracy']
plt.bar(accuracy_labels, accuracy_values, color=['blue', 'red'])
plt.ylabel('Accuracy')
plt.title('Train Accuracy vs Test Accuracy')
plt.ylim(0, 1)
plt.show();